# TRAIN MODEL MŨ BẢO HIỂM & BIỂN SỐ v3 (DỮ LIỆU ĐƯỜNG PHỐ MỚI)

### Hướng dẫn 3 bước cực nhanh trên Google Colab:
1. Vào menu: **Thời lượng chạy (Runtime)** -> **Thay đổi loại thời lượng chạy (Change runtime type)** -> Chọn **T4 GPU** -> Bấm **Lưu (Save)**.
2. Nhìn cột bên trái, bấm vào biểu tượng **Thư mục (Files)** -> Kéo thả 2 file từ máy tính vào đây:
   - `dataset_v3_train.zip` (ở thư mục gốc `D:\hoclamAI\giaothong\`)
   - `yolo11s_helmet_lp.pt` (ở thư mục `D:\hoclamAI\giaothong\models\`)
3. Bấm **Thời lượng chạy (Runtime)** -> **Chạy tất cả (Run all)**. Đợi ~3-4 phút model v3 sẽ tự động tải về máy bạn!

In [ ]:
# 1. Kiểm tra GPU và cài đặt thư viện
!nvidia-smi
!pip install -q ultralytics

In [ ]:
import os, glob, shutil, zipfile

# 2. Giải nén dataset vào /content/dataset_v3_street
zip_files = glob.glob('/content/**/dataset_v3_train*.zip', recursive=True)
assert len(zip_files) > 0, 'CHƯA TÌM THẤY file dataset_v3_train.zip! Vui lòng kéo thả file này vào cột Files bên trái!'

print(f'Đang giải nén: {zip_files[0]}...')
with zipfile.ZipFile(zip_files[0], 'r') as zf:
    zf.extractall('/content/dataset_v3_street')

# Tự động cập nhật data.yaml
yaml_files = glob.glob('/content/**/data.yaml', recursive=True)
assert len(yaml_files) > 0, 'Không tìm thấy data.yaml trong bộ dữ liệu!'
yaml_path = yaml_files[0]
folder = os.path.dirname(os.path.abspath(yaml_path))

with open(yaml_path, 'r', encoding='utf-8') as f:
    content = f.read()

lines = []
for line in content.split('\n'):
    if line.strip().startswith('path:'):
        lines.append(f"path: '{folder}'")
    else:
        lines.append(line)

with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print('Dataset v3 đã sẵn sàng tại:', folder)
print('data.yaml:', yaml_path)

In [ ]:
from ultralytics import YOLO
import glob

# 3. Nạp model gốc để Fine-tune
base_pts = glob.glob('/content/**/yolo11s_helmet_lp.pt', recursive=True)
base_model = base_pts[0] if base_pts else 'yolo11s.pt'
print(f'Sử dụng base model: {base_model}')

model = YOLO(base_model)

# Bắt đầu huấn luyện
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    lr0=0.001,
    degrees=10.0,
    perspective=0.0005,
    hsv_v=0.4,
    save=True,
    project='/content/runs_v3',
    name='helmet_v3'
)
print('HUẤN LUYỆN HOÀN TẤT!')

In [ ]:
import os, glob, shutil
from google.colab import files

# 4. Tải file weights tốt nhất về máy tính
best_weights = glob.glob('/content/runs_v3/**/best.pt', recursive=True)
if best_weights:
    best_file = best_weights[0]
    target_name = '/content/yolo11s_helmet_lp.pt'
    shutil.copyfile(best_file, target_name)
    print(f'Đang tải file model tốt nhất: {best_file}...')
    files.download(target_name)
else:
    print('Không tìm thấy file best.pt!')